Update image with cellular automaton
- find rules

## Import modules

In [1]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
from datetime import date
# from dataclasses import dataclass, field
# import itertools


# import 3rd-party modules
import cv2
import numpy as np
# from numba import njit
# import ray
# from pygifsicle import optimize


# import local modules
from core.utils.renderer.giffer import create_gif
from core.utils.renderer.videographer import create_video
from core.utils.project_manager import Project
from core.utils.renderer.resizer import get_interpolation
from core.utils.renderer.resizer import resize_with_pad, resize_with_crop

## Set up project

In [2]:
# create project
project = Project(project_dir="assets/images/mosaic/cellular_automaton")

## Define functions

In [160]:
def find_nearest_multiple(x, base):
    """
    Function to find the nearest multiple of base given an integer
    """
    return base * round(x/base)

def find_surrounding_cells(cell_yx, img_shape):
    """
    Function to find yx coords of surrounding cells in image.
    """
    
    # create empty list to store yx coords of surrounding cells
    surrounding_yxs = []

    # unpack cell yx coords
    y, x = cell_yx

    # set possible surrounding indexes
    surrounding_idxs = np.array([
    [y-1, x-1],
    [y-1, x],
    [y-1, x+1],
    [y, x-1],
    [y, x+1],
    [y+1, x-1],
    [y+1, x],
    [y+1, x+1],
    ])

    # get only surrounding indexes inside image
    for surrounding_idx in surrounding_idxs:
        s_y_idx, s_x_idx = surrounding_idx
        if not ((surrounding_idx < 0).any() or s_y_idx >= img_shape[0] or s_x_idx >= img_shape[1]):
            surrounding_yxs.append(surrounding_idx)
            
    return surrounding_yxs

def mean_automaton_rules(cell_hue, surrounding_cells_hues):
    """
    Function to define cellular automaton rules to update the current cell
    based on its state and states from surrounding cells
    """
    
    surrounding_cells_hues_mean = np.mean(surrounding_cells_hues)

    if cell_hue < surrounding_cells_hues_mean:
        return surrounding_cells_hues_mean

In [157]:
def run_cellular_automaton(
    dest_img, automaton_rules_fct, nb_rows, nb_cols, col_range=None,
    img_channel_to_modify=0 # hue by default
    ):
    """
    Function to run a cellular_automaton on an image
    """

    nb_updates = 0

    # get img height & width
    dest_img_height, dest_img_width = dest_img.shape[:2]

    # get best number of rows and cols to cover the whole img
    out_img_height = find_nearest_multiple(dest_img_height, nb_rows)
    out_img_width = find_nearest_multiple(dest_img_width, nb_cols)

    # resize image to recreate so that the grid can cover the whole image
    # get interpolation
    interpolation = get_interpolation(src_img_shape=(dest_img_height, dest_img_width), out_img_shape=(out_img_height, out_img_width))
    dest_img = cv2.resize(dest_img, (out_img_width, out_img_height), interpolation=interpolation)

    # get cell height & width
    cell_height, cell_width = out_img_height//nb_rows, out_img_width//nb_cols

    # get list of grid positions
    if col_range is not None:
        grid_positions = [(grid_y, grid_x) for grid_x in range(*col_range) for grid_y in range(nb_rows)]
        # out_img = np.zeros_like(dest_img[:,col_range[0]:col_range[1]])
    else:
        grid_positions = [(grid_y, grid_x) for grid_x in range(nb_cols) for grid_y in range(nb_rows)]
        # out_img = np.zeros_like(dest_img)

    # get index range of grid positions
    grid_positions_idxs = np.arange(len(grid_positions))
    # # choose random seed to recreate same shuffle or change it to see if you get better results
    # np.random.seed(1111)
    # shuffle grid positions (otherwise the first grids from top will get the best matching images)
    # np.random.shuffle(grid_positions_idxs)

    # convert bgr image to hsv
    dest_img = cv2.cvtColor(dest_img, cv2.COLOR_BGR2HSV)

    # iterate over each grid position
    for i in grid_positions_idxs:

        grid_y, grid_x = grid_positions[i]
        
        # get hues of region of interest in img to recreate
        y = grid_y * cell_height
        x = grid_x * cell_width
        dest_roi_hues = dest_img[y:y+cell_height, x:x+cell_width, img_channel_to_modify]
        
        # get mean of roi
        dest_roi_hues_mean = np.mean(dest_roi_hues)

        # find surrounding cells
        surrounding_cells_grid_yxs = find_surrounding_cells((grid_y, grid_x), (nb_rows, nb_cols))

        # create empty list to store hue values of surrounding cells
        surrounding_cells_hues = []

        # get surrounding cells rois
        for surrounding_cells_grid_yx in surrounding_cells_grid_yxs:

            s_grid_y, s_grid_x = surrounding_cells_grid_yx

            # get hues of surrounding regions of interest in img to recreate
            s_y = s_grid_y * cell_height
            s_x = s_grid_x * cell_width
            dest_s_roi_hues = dest_img[s_y:s_y+cell_height, s_x:s_x+cell_width, img_channel_to_modify]
        
            # get mean of surrounding roi
            dest_s_roi_hues_mean = np.mean(dest_s_roi_hues)

            # append to list of hue values of surrounding cells
            surrounding_cells_hues.append(dest_s_roi_hues_mean)

        # apply automaton_rules_fct
        dest_roi_hues_update = automaton_rules_fct(dest_roi_hues_mean, surrounding_cells_hues)

        # if cell updates, apply new hue values
        if dest_roi_hues_update is not None:
            dest_img[y:y+cell_height, x:x+cell_width, img_channel_to_modify] = int(dest_roi_hues_update)

            # record that there has been an update
            nb_updates += 1

    # convert bgr image to hsv
    dest_img = cv2.cvtColor(dest_img, cv2.COLOR_HSV2BGR)

    return nb_updates, dest_img

## Read images

In [3]:
# set dest image path (i.e path of image to recreate)
dest_img_path = Path("/Users/derrickvanfrausum/Pictures/JWT_Carina_Nebula.png")

# get dest image
dest_img = cv2.imread(str(dest_img_path))

## Define constants, variables & output image directory

In [5]:
NB_ROWS = 200
NB_COLS = 200
NB_IMGS = 200
NB_UPDATES_THRESHOLD = 100

# set channel to update
img_channel_to_modify = 0

# set output image directory
out_img_dir = f"random_{dest_img_path.stem}__{NB_ROWS*NB_COLS}_channel{img_channel_to_modify}"

# make output image directory
project.make_dir(out_img_dir)

## Run cellular automaton

In [165]:
# get current date
today = date.today().strftime("%Y%m%d")

out_img = dest_img.copy()

for i in range(0, NB_IMGS):

    # recreate image with mosaic of shape
    nb_updates, out_img = run_cellular_automaton(out_img, mean_automaton_rules, NB_ROWS, NB_COLS, 
    img_channel_to_modify=img_channel_to_modify)
    print(nb_updates)

    # stop writing images if no more updates
    if nb_updates < NB_UPDATES_THRESHOLD:
        break

    # set output image path
    out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{NB_ROWS*NB_COLS}_{today}_{i:09d}.jpg")

    # save output image & metadata
    cv2.imwrite(out_img_path, out_img)

23495
25970
26810
27038
27059
27048
27035
27024
27017
26972
26927
26901
26898
26930
26933
26872
26886
26844
26823
26743
26718
26685
26686
26631
26633
26601
26561
26534
26558
26534
26496
26537
26515
26459
26475
26468
26404
26433
26405
26341
26355
26393
26380
26342
26289
26283
26238
26257
26190
26248
26202
26195
26207
26162
26139
26128
26115
26148
26060
26038
26028
26051
26052
26036
26033
26036
25995
25982
25943
25991
25954
25975
25901
25908
25959
25976
25946
25917
25911
25900
25893
25914
25862
25856
25892
25858
25864
25862
25828
25834
25826
25805
25827
25867
25829
25786
25810
25773
25780
25778
25791
25797
25784
25784
25796
25764
25758
25795
25778
25755
25725
25753
25743
25753
25790
25753
25775
25762
25744
25714
25716
25724
25746
25703
25743
25742
25700
25712
25715
25733
25715
25687
25702
25692
25695
25686
25673
25702
25673
25692
25745
25704
25725
25692
25712
25666
25684
25693
25670
25705
25672
25741
25684
25747
25664
25683
25685
25678
25681
25692
25703
25693
25671
25679
25698
25688
2569

In [ ]:
for i in range(i, i+100):

    # recreate image with mosaic of shape
    nb_updates, out_img = run_cellular_automaton(out_img, mean_automaton_rules, NB_ROWS, NB_COLS, 
    img_channel_to_modify=img_channel_to_modify)
    print(nb_updates)

    # stop writing images if no more updates
    if nb_updates < NB_UPDATES_THRESHOLD:
        break

    # set output image path
    out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{NB_ROWS*NB_COLS}_{today}_{i:09d}.jpg")

    # save output image & metadata
    cv2.imwrite(out_img_path, out_img)

In [18]:
# add input img into the list
img_path_list = [str(dest_img_path)]

# put the first 100 iterations as there are lots of updates (toDo: get all iterations and speed up gif when few changes
# to make the gif changes more striking visually)
img_path_list.extend(project.get_img_path_list(project.out_img_dir_dict[out_img_dir])[:100])

In [20]:
# get img height & width
dest_img_height, dest_img_width = dest_img.shape[:2]

# set gif path
# gif_path = project.project_dir / f"{out_img_dir}.gif"
gif_path = project.project_dir / f"{out_img_dir}.gif"

# create gif
# create_gif(img_dir=project.out_img_dir_dict[out_img_dir], out_path=gif_path, out_img_shape=(int(dest_img_height//5),int(dest_img_width//5)),
# sort_img_list=True, duplicate_start_img_amount=0, duplicate_end_img_amount=0,
# optimize_gif=False)
create_gif(img_path_list=img_path_list, out_path=gif_path, out_img_shape=(int(dest_img_height//4),int(dest_img_width//4)),
sort_img_list=True, duplicate_start_img_amount=5, duplicate_end_img_amount=0,
optimize_gif=False)

In [166]:
NB_ROWS = dest_img.shape[0]
NB_COLS = dest_img.shape[1]

# set channel to update
img_channel_to_modify = 0

# set output image directory
out_img_dir = f"random_{dest_img_path.stem}__{NB_ROWS*NB_COLS}_channel{img_channel_to_modify}"

# make output image directory
project.make_dir(out_img_dir)

In [172]:
# get current date
today = date.today().strftime("%Y%m%d")

nb_updates_threshold = 1000

out_img = dest_img.copy()

counter = 0
while nb_updates > nb_updates_threshold:
    counter += 1

    # recreate image with mosaic of shape
    nb_updates, out_img = run_cellular_automaton(out_img, mean_automaton_rules, NB_ROWS, NB_COLS, 
    img_channel_to_modify=img_channel_to_modify)


# set output image path
out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{NB_ROWS*NB_COLS}_{today}_{counter}.jpg")

# save output image & metadata
cv2.imwrite(out_img_path, out_img)

KeyboardInterrupt: 

In [170]:
# set output image path
out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{NB_ROWS*NB_COLS}_{today}_{counter}.jpg")

# save output image & metadata
cv2.imwrite(out_img_path, out_img)

True

In [173]:
counter

165

## run cellular automaton on random image

In [126]:
# set dest image path (i.e path of image to recreate)
dest_img_path = Path("random_noise")

NB_ROWS = 1000
NB_COLS = 1000

# get random image
dest_img = np.random.randint(0, 255, size=(NB_ROWS, NB_COLS, 3), dtype=np.uint8)

# set channel to update
img_channel_to_modify = 0

# set output image directory
out_img_dir = f"random_{dest_img_path.stem}__{NB_ROWS*NB_COLS}_channel{img_channel_to_modify}"

# make output image directory
project.make_dir(out_img_dir)

# get current date
today = date.today().strftime("%Y%m%d")

# set input image path
i = 0
in_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{NB_ROWS*NB_COLS}_{today}_{i:09d}.jpg")

# save input image & metadata
cv2.imwrite(in_img_path, dest_img)

True

In [127]:
NB_IMGS = 200
nb_updates_threshold = 1

out_img = dest_img.copy()

for i in range(1, NB_IMGS):

    # recreate image with mosaic of shape
    nb_updates, out_img = run_cellular_automaton(out_img, mean_automaton_rules, NB_ROWS, NB_COLS, 
    img_channel_to_modify=img_channel_to_modify)
    print(nb_updates)

    # stop writing images if no more updates
    if nb_updates < nb_updates_threshold:
        break

    # set output image path
    out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"{NB_ROWS*NB_COLS}_{today}_{i:09d}.jpg")

    # save output image & metadata
    cv2.imwrite(out_img_path, out_img)

588465
717225
781323
820405
847579
867737
882506
893278


KeyboardInterrupt: 